# Persona Knowledge Graph — Colab

Free, no local storage. This notebook downloads the latest Kuzu graph built by
the `graph-build` GitHub Action and runs **segmentation**, **similarity**, and
**GraphRAG** queries.

Run cells top to bottom. Nothing is installed on your machine — it all lives in
Colab's ephemeral runtime.

In [ ]:
REPO = "sargupta/persona-engine"   # change if you fork
TAG = "graph-latest"
!pip -q install kuzu pyarrow pyvis

In [ ]:
import urllib.request, tarfile, os
url = f"https://github.com/{REPO}/releases/download/{TAG}/persona_graph.kuzu.tar.gz"
print("downloading", url)
urllib.request.urlretrieve(url, "persona_graph.kuzu.tar.gz")
with tarfile.open("persona_graph.kuzu.tar.gz") as t:
    t.extractall(".")
DB = "persona_graph.kuzu"
print("extracted ->", DB, "(", round(os.path.getsize('persona_graph.kuzu.tar.gz')/1e6,1), "MB archive )")
# query helpers (same module used by the CLI)
urllib.request.urlretrieve(f"https://raw.githubusercontent.com/{REPO}/main/graph/query_kuzu.py", "query_kuzu.py")

In [ ]:
import query_kuzu
conn = query_kuzu.open_conn(DB)
# sanity check
def one(q):
    r = conn.execute(q)
    return r.get_next() if r.has_next() else None
print("personas:", one("MATCH (p:Persona) RETURN count(p)"))
print("occupations:", one("MATCH (o:Occupation) RETURN count(o)"))

## 1. Segmentation

In [ ]:
import json
seg = query_kuzu.segment(conn, tier="manual", community="OBC",
                         state="Uttar Pradesh", min_scarcity=0.6, limit=5)
print("cohort size:", seg["total"])
print(json.dumps(seg["sample"], indent=2, ensure_ascii=False, default=str))
PID = seg["sample"][0]["id"]   # use one as a similarity seed
PID

## 2. Behavioral similarity (vector KNN)

In [ ]:
sim = query_kuzu.similar(conn, PID, k=5)
print(json.dumps(sim, indent=2, ensure_ascii=False, default=str))

## 3. GraphRAG context bundle (for an LLM)

In [ ]:
ctx = query_kuzu.rag_context(conn, PID, k=3)
print(json.dumps(ctx, indent=2, ensure_ascii=False, default=str))

## 4. Visualize a cohort subgraph

In [ ]:
from pyvis.network import Network
net = Network(height="600px", notebook=True, cdn_resources="in_line")
rows, r = [], conn.execute(
    "MATCH (p:Persona)-[:HOLDS_VALUE]->(v:Value) "
    "WHERE p.scarcity_state >= 0.6 RETURN p.name, v.name LIMIT 60")
while r.has_next():
    rows.append(r.get_next())
for pname, vname in rows:
    net.add_node(pname, label=pname, color="#5b8def")
    net.add_node(vname, label=vname, color="#f4a261", shape="box")
    net.add_edge(pname, vname)
net.show("cohort.html")
from IPython.display import HTML
HTML(open("cohort.html").read())